# Complete Anime Character Organization Pipeline

This master notebook runs the entire end-to-end anime character organization workflow:
1. **Dataset Audit**: Scan image collection, validate file readability, and detect exact (SHA-256) and perceptual (pHash) duplicates.
2. **Crop Preparation**: Detect anime heads and bodies via `imgutils`, apply margin padding, and extract identity crops.
3. **Embedding Extraction**: Extract character similarity embeddings via deep learning model (`CCIP`).
4. **HDBSCAN Clustering**: Group character embeddings using density clustering with optimized parameters and noise reassignment.
5. **Folder Materialization**: Non-destructively materialize organized character folders via hardlinks, copies, or symlinks.
6. **Semantic Naming**: Infer character identity tags using anime tagger models (`WD14` / `PixAI`) and apply agreed names.

> **Note**: Change the parameters in the first cell below to configure your run.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

# ==============================================================================
# PIPELINE PARAMETERS & CONFIGURATION
# Change these parameters to configure your run.
# ==============================================================================

# --- 1. Paths & Directories ---
# Source folder containing the raw anime images to organize
INPUT_DIR = Path("/home/ssparzival/Imágenes/Ente Photos/Twitter/").expanduser().resolve()

# Workspace directory where pipeline runs and organized output will be stored
PROJECT_DIR = Path("./anime_character_pipeline").expanduser().resolve()

# --- 2. Stage 1: Audit & Duplicate Detection ---
# Maximum Hamming distance on 64-bit pHash for near-duplicate candidate detection
PHASH_THRESHOLD = 6

# --- 3. Stage 2: Face/Body Detection & Crop Preparation ---
# Keep False so distinct images/poses are NOT discarded before clustering
USE_PERCEPTUAL_REPRESENTATIVES = False

# Detection confidence thresholds and bounding box padding
HEAD_CONF_THRESHOLD = 0.35
PERSON_CONF_THRESHOLD = 0.30
HEAD_PADDING_RATIO = 0.25   # Generous padding around hair/accessories for CCIP
CROP_OUTPUT_SIZE = (512, 512)
CROP_FORMAT = "JPEG"

# --- 4. Stage 3: CCIP Character Similarity Embeddings ---
CCIP_MODEL = "ccip-caformer-24-randaug-pruned"
CCIP_IMAGE_SIZE = 384
BATCH_SIZE = 16

# --- 5. Stage 4: Optimized HDBSCAN Clustering ---
# Researched & optimized parameters to prevent excessive noise classification:
MIN_CLUSTER_SIZE = 2             # Allows pairs and triplets of characters to form clusters (previously 5)
MIN_SAMPLES = 1                  # Eliminates harsh reachability distance penalty on small clusters (previously 4)
CLUSTER_SELECTION_EPSILON = 0.50   # Merges points with high visual similarity (CCIP threshold ~0.50)
CLUSTER_SELECTION_METHOD = "eom"

# Reassign borderline noise points within threshold distance to their nearest cluster centroid
REASSIGN_NOISE = True
MAX_REASSIGN_DISTANCE = 0.55     # Max distance to cluster centroid to assign from noise

# Keep character images unified in one folder instead of splitting into _review folders
SEPARATE_REVIEW_FOLDERS = False

# --- 6. Stage 5: Non-Destructive Materialization ---
# Materialization mode: "hardlink" (instant, 0 extra disk), "copy", or "symlink"
MATERIALIZATION_MODE = "hardlink"
ALLOW_COPY_FALLBACK = True

# --- 7. Stage 6: Semantic Character Naming (Anime Tagger) ---
RUN_TAGGING = True               # Set to True to infer character names using WD14/PixAI
MIN_CHARACTER_SCORE = 0.65       # Minimum model confidence score for a character tag
MIN_NAME_SHARE = 0.30            # Minimum fraction of cluster images with the top tag
MIN_TOP_MARGIN = 0.05            # Score margin over second candidate
APPLY_FOLDER_RENAME = True       # Apply the proposed character names to output folders

print("=" * 70)
print("CONFIGURATION LOADED SUCCESSFULLY")
print(f"Input Directory:        {INPUT_DIR}")
print(f"Project Directory:      {PROJECT_DIR}")
print(f"HDBSCAN Parameters:     min_cluster_size={MIN_CLUSTER_SIZE}, min_samples={MIN_SAMPLES}, epsilon={CLUSTER_SELECTION_EPSILON}")
print(f"Noise Reassignment:     enabled={REASSIGN_NOISE}, max_distance={MAX_REASSIGN_DISTANCE}")
print(f"Folder Organization:    separate_review_folders={SEPARATE_REVIEW_FOLDERS}, mode={MATERIALIZATION_MODE}")
print(f"Semantic Tagging:       enabled={RUN_TAGGING}")
print("=" * 70)


## Stage 1: Dataset Audit, Integrity Validation, and Duplicate Index


In [ ]:
from anime_character_organizer import run_dataset_audit

print(">>> Running Stage 1: Dataset Audit...")
audit_res = run_dataset_audit(
    input_dir=INPUT_DIR,
    project_dir=PROJECT_DIR,
    phash_threshold=PHASH_THRESHOLD,
    show_progress=True,
)

audit_summary = audit_res["summary"]
print()
print("[Stage 1 Completed]")
print(f"Run Directory:              {audit_res['run_dir']}")
print(f"Total Scanned Files:        {audit_summary.get('total_scanned_files', audit_summary.get('total_candidate_files'))}")
print(f"Valid Image Files:          {audit_summary.get('valid_images', len(audit_res.get('valid_df', [])))}")
print(f"Invalid / Corrupted Files:  {audit_summary.get('invalid_images', len(audit_res.get('invalid_df', [])))}")
print(f"Exact Duplicate Groups:     {audit_summary.get('exact_duplicate_groups', 0)}")
print(f"Perceptual Duplicate Pairs: {audit_summary.get('perceptual_duplicate_candidate_pairs', 0)}")

invalid_df = audit_res.get("invalid_images_df", audit_res.get("invalid_df"))
if invalid_df is not None and not invalid_df.empty:
    display(invalid_df.head(5))


## Stage 2: Anime Face/Body Detection and Crop Preparation


In [ ]:
from anime_character_organizer import run_crop_preparation

print(">>> Running Stage 2: Crop Preparation...")
crop_res = run_crop_preparation(
    project_dir=PROJECT_DIR,
    previous_run_dir=audit_res["run_dir"],
    use_perceptual_representatives=USE_PERCEPTUAL_REPRESENTATIVES,
    head_conf_threshold=HEAD_CONF_THRESHOLD,
    person_conf_threshold=PERSON_CONF_THRESHOLD,
    head_padding_ratio=HEAD_PADDING_RATIO,
    crop_output_size=CROP_OUTPUT_SIZE,
    crop_format=CROP_FORMAT,
    show_progress=True,
)

crop_summary = crop_res["summary"]
print()
print("[Stage 2 Completed]")
print(f"Run Directory:        {crop_res['run_dir']}")
print(f"Total Crops Created:  {crop_summary.get('successful_crops', 0)}")
print(f"Crop Errors:          {crop_summary.get('crop_errors', 0)}")
print(f"Flagged for Review:   {crop_summary.get('review_required', 0)}")

crop_manifest = crop_res.get("crop_manifest_df", crop_res.get("review_df"))
if crop_manifest is not None and "selected_region_type" in crop_manifest.columns:
    region_counts = crop_manifest["selected_region_type"].value_counts()
    print()
    print("Selected Region Breakdown:")
    display(region_counts.to_frame(name="Count"))


## Stage 3: CCIP Character Similarity Feature Extraction


In [ ]:
from anime_character_organizer import run_embedding_extraction

print(">>> Running Stage 3: CCIP Feature Extraction...")
embed_res = run_embedding_extraction(
    project_dir=PROJECT_DIR,
    previous_run_dir=crop_res["run_dir"],
    ccip_model=CCIP_MODEL,
    ccip_image_size=CCIP_IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    show_progress=True,
)

embed_summary = embed_res["summary"]
embeddings_mat = embed_res.get("embeddings", embed_res.get("normalized_embeddings"))
print()
print("[Stage 3 Completed]")
print(f"Run Directory:           {embed_res['run_dir']}")
if embeddings_mat is not None:
    print(f"Embeddings Matrix Shape: {embeddings_mat.shape}")
print(f"Successful Embeddings:   {embed_summary.get('successful_embeddings', 0)}")
print(f"Failed Extractions:      {embed_summary.get('failed_embeddings', 0)}")


## Stage 4: HDBSCAN Clustering with Researched Parameters & Diagnostics


In [ ]:
from anime_character_organizer import run_clustering

print(">>> Running Stage 4: HDBSCAN Clustering & Diagnostics...")
cluster_res = run_clustering(
    project_dir=PROJECT_DIR,
    previous_run_dir=embed_res["run_dir"],
    min_cluster_size=MIN_CLUSTER_SIZE,
    min_samples=MIN_SAMPLES,
    cluster_selection_epsilon=CLUSTER_SELECTION_EPSILON,
    cluster_selection_method=CLUSTER_SELECTION_METHOD,
    reassign_noise=REASSIGN_NOISE,
    max_reassign_distance=MAX_REASSIGN_DISTANCE,
    separate_review_folders=SEPARATE_REVIEW_FOLDERS,
    show_progress=True,
)

cluster_summary = cluster_res["summary"]
emb_count = max(1, cluster_summary.get('embedding_count', 1))
print()
print("[Stage 4 Completed]")
print(f"Run Directory:           {cluster_res['run_dir']}")
print(f"Discovered Clusters:     {cluster_summary.get('estimated_clusters', 0)}")
print(f"Clustered Images:        {cluster_summary.get('clustered_points', 0)} ({cluster_summary.get('clustered_points', 0) / emb_count:.1%})")
print(f"Reassigned from Noise:   {cluster_summary.get('reassigned_points', 0)}")
print(f"Remaining Noise Images:  {cluster_summary.get('noise_points', 0)} ({cluster_summary.get('noise_points', 0) / emb_count:.1%})")
print(f"Manual Review Points:    {cluster_summary.get('manual_review_points', 0)}")

summary_df = cluster_res.get("cluster_summary_df")
if summary_df is not None and not summary_df.empty:
    top_clusters = summary_df[~summary_df["is_noise_cluster"]].head(10)
    print()
    print("Top Discovered Character Clusters:")
    display(top_clusters[["cluster_label", "cluster_folder_stub", "image_count", "mean_probability", "mean_distance_to_centroid"]])

cluster_size_plot = cluster_res["run_dir"] / "figures" / "cluster_size_distribution.png"
if cluster_size_plot.exists():
    display(Image(filename=cluster_size_plot.as_posix()))


## Stage 5: Non-Destructive Folder Materialization


In [ ]:
from anime_character_organizer import run_folder_materialization

print(">>> Running Stage 5: Non-Destructive Folder Materialization...")
materialize_res = run_folder_materialization(
    project_dir=PROJECT_DIR,
    previous_run_dir=cluster_res["run_dir"],
    materialization_mode=MATERIALIZATION_MODE,
    allow_hardlink_fallback_to_copy=ALLOW_COPY_FALLBACK,
    show_progress=True,
)

mat_summary = materialize_res["summary"]
print()
print("[Stage 5 Completed]")
print(f"Organized Output Directory: {materialize_res['final_output_dir']}")
print(f"Total Folders Created:       {mat_summary.get('total_folders', 0)}")
print(f"Total Files Materialized:    {mat_summary.get('total_files', 0)}")
print(f"Successful Operations:       {mat_summary.get('successful_operations', 0)}")

if "operation_counts_df" in materialize_res:
    display(materialize_res["operation_counts_df"])

if "folder_counts_df" in materialize_res:
    display(materialize_res["folder_counts_df"].head(10))


## Stage 6: Semantic Cluster Naming with Anime Taggers (Optional)


In [ ]:
if RUN_TAGGING:
    from anime_character_organizer import run_cluster_naming

    print(">>> Running Stage 6: Semantic Character Naming with Anime Tagger...")
    naming_res = run_cluster_naming(
        project_dir=PROJECT_DIR,
        previous_clustering_run_dir=cluster_res["run_dir"],
        previous_materialization_run_dir=materialize_res["run_dir"],
        min_character_score=MIN_CHARACTER_SCORE,
        min_name_share=MIN_NAME_SHARE,
        min_top_margin=MIN_TOP_MARGIN,
        create_named_output=APPLY_FOLDER_RENAME,
        named_output_mode=MATERIALIZATION_MODE,
        show_progress=True,
    )

    naming_summary = naming_res["summary"]
    print()
    print("[Stage 6 Completed]")
    print(f"Run Directory:           {naming_res['run_dir']}")
    print(f"Candidate Clusters:      {naming_summary.get('candidate_clusters', naming_summary.get('total_clusters_evaluated', 0))}")
    print(f"Successfully Named:      {naming_summary.get('named_clusters', naming_summary.get('accepted_character_names', 0))}")
    print(f"Remaining Unknown:       {naming_summary.get('unnamed_clusters', 0)}")

    named_out = naming_res.get("named_output_dir") or naming_summary.get("named_output_dir")
    if APPLY_FOLDER_RENAME and named_out:
        print()
        print(f"Final Named Output Directory: {named_out}")

    preview_df = naming_res.get("name_suggestions_df", naming_res.get("cluster_name_suggestions_df"))
    if preview_df is not None and not preview_df.empty:
        display(preview_df[["cluster_label", "original_folder", "suggested_folder", "is_named", "top_tag", "top_score", "top_share"]].head(15))
else:
    print("Stage 6 semantic naming skipped by configuration.")


## Pipeline Execution Summary & Output Inspection


In [ ]:
print("=" * 80)
print("ANIME CHARACTER ORGANIZATION PIPELINE COMPLETED SUCCESSFULLY!")
print("=" * 80)
print(f"Input Directory:              {INPUT_DIR}")
print(f"Audited Images:               {audit_summary.get('valid_images', 0)}")
print(f"Extracted Crops:              {crop_summary.get('successful_crops', 0)}")
print(f"Identified Clusters:          {cluster_summary.get('estimated_clusters', 0)}")
cl_pts = cluster_summary.get('clustered_points', 0)
emb_cnt = max(1, cluster_summary.get('embedding_count', 1))
print(f"Clustering Efficiency:        {cl_pts}/{emb_cnt} ({cl_pts / emb_cnt:.1%}) classified")
noise_pts = cluster_summary.get('noise_points', 0)
print(f"Noise Reduction:              {noise_pts} remaining unclustered ({noise_pts / emb_cnt:.1%})")
print(f"Non-Destructive Output:       {materialize_res['final_output_dir']}")
named_out_dir = naming_res.get('named_output_dir') if RUN_TAGGING and 'naming_res' in locals() else None
if named_out_dir:
    print(f"Semantically Named Output:    {named_out_dir}")
print("=" * 80)
